# 05 - Stewardship
# Enterprise BFSI Client MDM Platform


## Purpose:
   1. Read Entity Resolution match results
   2. Build the stewardship review queue
   3. Create stewardship cases for REVIEW records
   4. Capture explicit steward decisions
   5. Preserve an auditable decision history
   6. Produce approved match relationships for downstream MDM

 Decision model:
   
   MATCH          → eligible for automatic clustering
   
   REVIEW         → requires stewardship
   
   APPROVE_MATCH  → eligible for clustering
   
   REJECT_MATCH   → excluded from clustering
   
   NO_MATCH       → remains unresolved

 IMPORTANT:
   - No ground truth is used for operational decisions.
   - REVIEW records are never merged automatically.
   - Every stewardship decision must be explainable and auditable.

##Load Review Queue

In [0]:
from pyspark.sql import functions as F

CATALOG = "bfsi_mdm_dev"

review_queue = spark.table(
    f"{CATALOG}.mdm.review_queue"
)

print(
    f"Review cases: {review_queue.count():,}"
)

Review cases: 159,560


##Create deterministic case IDs

In [0]:
cases = (
    review_queue
    .withColumn(
        "case_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("record_key_a"),
                F.col("record_key_b")
            ),
            256
        )
    )
)

##Add workflow fields

In [0]:
stewardship_cases = (
    cases
    .withColumn(
        "status",
        F.lit("OPEN")
    )
    .withColumn(
        "steward_id",
        F.lit(None).cast("string")
    )
    .withColumn(
        "decision",
        F.lit(None).cast("string")
    )
    .withColumn(
        "decision_reason",
        F.lit(None).cast("string")
    )
    .withColumn(
        "decision_timestamp",
        F.lit(None).cast("timestamp")
    )
    .withColumn(
        "created_at",
        F.current_timestamp()
    )
)

##Persist the stewardship case table

In [0]:
(
    stewardship_cases
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.mdm.stewardship_cases"
    )
)

##Rejecting a review

In [0]:
display(
    spark.table(
        "bfsi_mdm_dev.mdm.stewardship_cases"
    )
    .select(
        "case_id",
        "record_key_a",
        "record_key_b",
        "match_score",
        "matching_rule",
        "status"
    )
    .limit(1)
)

case_id,record_key_a,record_key_b,match_score,matching_rule,status
91bc2e588369057d3abeb308a99217cfd826bef818935f7792df449337545377,CRM::CRM00618369,KYC::KYC00760578,82.32,fuzzy_weighted,OPEN


In [0]:
%sql
UPDATE bfsi_mdm_dev.mdm.stewardship_cases
SET
    status = 'RESOLVED',
    steward_id = 'demo_steward',
    decision = 'REJECT_MATCH',
    decision_reason = 'Conflicting identity evidence indicates different clients.',
    decision_timestamp = current_timestamp()
WHERE case_id = '91bc2e588369057d3abeb308a99217cfd826bef818935f7792df449337545377';

num_affected_rows
1


##Create an auditable decisions table

In [0]:
stewardship_decisions = (
    spark.table(
        f"{CATALOG}.mdm.stewardship_cases"
    )
    .filter(
        F.col("status")
        ==
        "RESOLVED"
    )
    .select(
        "case_id",

        "record_key_a",
        "record_key_b",

        "source_system_a",
        "source_id_a",

        "source_system_b",
        "source_id_b",

        "match_score",
        "matching_rule",

        "steward_id",
        "decision",
        "decision_reason",
        "decision_timestamp"
    )
)

In [0]:
(
    stewardship_decisions
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        f"{CATALOG}.mdm.stewardship_decisions"
    )
)

In [0]:
decisions = spark.table(
    "bfsi_mdm_dev.mdm.stewardship_decisions"
)

print("Resolved stewardship decisions:", decisions.count())

display(
    decisions
    .select(
        "case_id",
        "record_key_a",
        "record_key_b",
        "match_score",
        "matching_rule",
        "steward_id",
        "decision",
        "decision_reason",
        "decision_timestamp"
    )
)

Resolved stewardship decisions: 2


case_id,record_key_a,record_key_b,match_score,matching_rule,steward_id,decision,decision_reason,decision_timestamp
ab23452e74b8f641feb7082a1a91bfe1e1ccacd531a3099204c285aab53ad969,CORE::C00240191,CRM::CRM00164741,81.76,fuzzy_weighted,demo_steward,APPROVE_MATCH,Same client based on strong supporting identity evidence.,2026-09-27T10:08:05.902119Z
91bc2e588369057d3abeb308a99217cfd826bef818935f7792df449337545377,CRM::CRM00618369,KYC::KYC00760578,82.32,fuzzy_weighted,demo_steward,REJECT_MATCH,Conflicting identity evidence indicates different clients.,2026-09-27T10:09:55.769693Z
